In [ ]:
import sys
import os
from pathlib import Path
search_paths = [Path.cwd(), Path.cwd().parent, Path('/mnt/data')]
for p in search_paths:
    p_str = str(p.resolve())
    if p.exists() and p_str not in sys.path:
        sys.path.append(p_str)
print('cwd:', Path.cwd())
print('import search paths added:')
for p in search_paths:
    print(' -', p.resolve())


In [ ]:
from IPython.display import display
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
import segmentation_models_pytorch as smp
from fastai.losses import DiceLoss, CrossEntropyLossFlat
from preprocessing import TunnelDataPipeline
from utils import save_training_history, save_prediction_overlap, val_loop, EarlyStopping


In [ ]:
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cudnn available:', torch.backends.cudnn.is_available())
n_gpu = torch.cuda.device_count()
print(f'Total GPUs available: {n_gpu}')
for i in range(n_gpu):
    print(f'Device {i}: {torch.cuda.get_device_name(i)}')
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} device')


In [ ]:
def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path('/mnt/data')]
    for p in candidates:
        if (p / 'TACK_Tunnel_Data').exists():
            return p.resolve()
    return Path.cwd().parent.resolve()

def baseline_model_pipeline(base_dir, dict_files, bs=16, img_size=512, use_custom_stats=False):
    base_dir = Path(base_dir)
    dataset_folder = base_dir / 'TACK_Tunnel_Data'
    csv_source_dir = dataset_folder / '2_model_input'
    raw_mask_dir = dataset_folder / '3_mask'
    if not dataset_folder.exists():
        raise FileNotFoundError(f'Could not find {dataset_folder}. Run dataset_download.py or set base_dir to the project root.')
    pipeline = TunnelDataPipeline(base_dir=str(dataset_folder), original_mask_dir=str(raw_mask_dir))
    train_files = dict_files['train_files']
    val_files = dict_files['val_files']
    test_files = dict_files['test_files']
    print('Loading CSV metadata...')
    df_train_val, df_test = pipeline.load_csv_data(csv_source_dir=str(csv_source_dir), train_files=train_files, val_files=val_files, test_files=test_files)
    if df_train_val.empty:
        raise ValueError(f'No train/validation rows were loaded from {csv_source_dir}. Check the CSV file names.')
    print('Sanitizing training and validation masks...')
    df_train_val_ready = pipeline.sanitize_masks(df_train_val, class_pixel_value=40)
    print('Sanitizing test masks...')
    df_test_ready = pipeline.sanitize_masks(df_test, class_pixel_value=40)
    if use_custom_stats:
        custom_stats = pipeline.calculate_training_stats(df_train_val_ready)
    else:
        custom_stats = None
    print('Generating DataLoaders...')
    train_dl, val_dl, test_dl = pipeline.get_dataloaders(train_val_df=df_train_val_ready, test_df=df_test_ready, bs=bs, img_size=img_size, custom_stats=custom_stats)
    print('\nPipeline ready:')
    print(f' - Training batches:   {len(train_dl)}')
    print(f' - Validation batches: {len(val_dl)}')
    print(f' - Testing batches:    {len(test_dl)}')
    return (train_dl, val_dl, test_dl, custom_stats)


In [ ]:
base_dir = find_project_root()
print('project root:', base_dir)
multi_domain_config = {'train_files': ['TA_train.csv', 'TB_train.csv', 'TC_train.csv'], 'val_files': ['TA_val.csv', 'TB_val.csv', 'TC_val.csv'], 'test_files': ['TA_test.csv', 'TB_test.csv', 'TC_test.csv']}
all_experiments = [({'train_files': ['TA_train.csv'], 'val_files': ['TA_val.csv'], 'test_files': ['TA_test.csv']}, 'Single-TA'), ({'train_files': ['TB_train.csv'], 'val_files': ['TB_val.csv'], 'test_files': ['TB_test.csv']}, 'Single-TB'), ({'train_files': ['TC_train.csv'], 'val_files': ['TC_val.csv'], 'test_files': ['TC_test.csv']}, 'Single-TC'), (multi_domain_config, 'Multi-Domain'), ({'train_files': ['TA_train.csv', 'TB_train.csv'], 'val_files': ['TA_val.csv', 'TB_val.csv'], 'test_files': ['TC_test.csv']}, 'Shift-TA_TB-to-TC_10pct'), ({'train_files': ['TA_train.csv', 'TB_train.csv'], 'val_files': ['TA_val.csv', 'TB_val.csv'], 'test_files': ['TC_train.csv', 'TC_val.csv', 'TC_test.csv']}, 'Shift-TA_TB-to-TC_100pct'), ({'train_files': ['TA_train.csv', 'TC_train.csv'], 'val_files': ['TA_val.csv', 'TC_val.csv'], 'test_files': ['TB_test.csv']}, 'Shift-TA_TC-to-TB_10pct'), ({'train_files': ['TA_train.csv', 'TC_train.csv'], 'val_files': ['TA_val.csv', 'TC_val.csv'], 'test_files': ['TB_train.csv', 'TB_val.csv', 'TB_test.csv']}, 'Shift-TA_TC-to-TB_100pct'), ({'train_files': ['TB_train.csv', 'TC_train.csv'], 'val_files': ['TB_val.csv', 'TC_val.csv'], 'test_files': ['TA_test.csv']}, 'Shift-TB_TC-to-TA_10pct'), ({'train_files': ['TB_train.csv', 'TC_train.csv'], 'val_files': ['TB_val.csv', 'TC_val.csv'], 'test_files': ['TA_train.csv', 'TA_val.csv', 'TA_test.csv']}, 'Shift-TB_TC-to-TA_100pct')]
RUN_ALL_EXPERIMENTS = True
experiments = all_experiments if RUN_ALL_EXPERIMENTS else [(multi_domain_config, 'Multi-Domain')]
print(f'Number of experiments selected: {len(experiments)}')


In [ ]:
class TTDCombinedLoss(nn.Module):

    def __init__(self, ce_weight_tensor, w_ce=0.5, w_dice=0.5):
        super().__init__()
        self.w_ce, self.w_dice = (w_ce, w_dice)
        self.ce_loss = CrossEntropyLossFlat(weight=ce_weight_tensor, axis=1)
        self.dice_loss = DiceLoss(axis=1)

    def forward(self, pred, targ):
        pred_tensor = pred.as_subclass(torch.Tensor)
        targ_tensor = targ.as_subclass(torch.Tensor).long()
        ce = self.ce_loss(pred_tensor, targ_tensor)
        dice = self.dice_loss(pred_tensor, targ_tensor)
        return self.w_ce * ce + self.w_dice * dice


In [ ]:
def freeze_encoder_for_feature_extraction(model):
    for p in model.encoder.parameters():
        p.requires_grad = False
    for p in model.decoder.parameters():
        p.requires_grad = True
    for p in model.segmentation_head.parameters():
        p.requires_grad = True
    if getattr(model, 'classification_head', None) is not None:
        for p in model.classification_head.parameters():
            p.requires_grad = True
    return model

def build_feature_extraction_unet(encoder_name='resnet34', encoder_weights='imagenet', classes=2):
    model = smp.Unet(encoder_name=encoder_name, encoder_weights=encoder_weights, classes=classes)
    return freeze_encoder_for_feature_extraction(model)

def parameter_report(model):
    total = sum((p.numel() for p in model.parameters()))
    trainable = sum((p.numel() for p in model.parameters() if p.requires_grad))
    frozen = total - trainable
    enc_trainable = sum((p.numel() for p in model.encoder.parameters() if p.requires_grad))
    dec_trainable = sum((p.numel() for p in model.decoder.parameters() if p.requires_grad))
    head_trainable = sum((p.numel() for p in model.segmentation_head.parameters() if p.requires_grad))
    print(f'Total parameters:             {total:,}')
    print(f'Frozen parameters:            {frozen:,}')
    print(f'Trainable parameters:         {trainable:,}')
    print(f'Encoder trainable parameters: {enc_trainable:,}')
    print(f'Decoder trainable parameters: {dec_trainable:,}')
    print(f'Head trainable parameters:    {head_trainable:,}')

def assert_encoder_is_frozen(model):
    bad = [name for name, p in model.encoder.named_parameters() if p.requires_grad]
    if bad:
        raise AssertionError(f'Encoder is not fully frozen. First trainable encoder parameter: {bad[0]}')
    print('Encoder freeze check passed: all encoder parameters have requires_grad=False.')

def trainable_parameters(model):
    return (p for p in model.parameters() if p.requires_grad)


In [ ]:
def train_loop_feature_extractor(model, device, dataloader, loss_fn, optimizer, scheduler=None):
    model.train()
    model.encoder.eval()
    running_loss = 0.0
    total_samples = 0
    for X_batch, y_batch in dataloader:
        X_batch = torch.as_tensor(X_batch).to(device)
        y_batch = torch.as_tensor(y_batch).to(device).long()
        optimizer.zero_grad(set_to_none=True)
        output = model(X_batch)
        loss = loss_fn(output.as_subclass(torch.Tensor), y_batch)
        loss.backward()
        optimizer.step()
        if scheduler and isinstance(scheduler, torch.optim.lr_scheduler.OneCycleLR):
            scheduler.step()
        batch_size = X_batch.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size
    if scheduler and (not isinstance(scheduler, torch.optim.lr_scheduler.OneCycleLR)):
        if not isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step()
    return running_loss / total_samples

def epochs_feature_extractor(model, model_name, device, train_dl, val_dl, loss_fn, optimizer, num_epoch, scheduler=None, patience=10, save_dir='models'):
    os.makedirs(save_dir, exist_ok=True)
    early_stopper = EarlyStopping(patience=patience)
    model = model.to(device)
    assert_encoder_is_frozen(model)
    best_iou = -float('inf')
    history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_f1': []}
    pbar = tqdm(range(num_epoch), desc=f'  → {model_name}', leave=False)
    for epoch in pbar:
        t_loss = train_loop_feature_extractor(model, device, train_dl, loss_fn, optimizer, scheduler)
        v_loss, v_iou, v_f1 = val_loop(model, device, val_dl, loss_fn)
        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['val_iou'].append(v_iou)
        history['val_f1'].append(v_f1)
        pbar.set_postfix({'IoU': f'{v_iou:.4f}', 'F1': f'{v_f1:.4f}'})
        checkpoint_status = ''
        if v_iou > best_iou:
            best_iou = v_iou
            torch.save(model.state_dict(), os.path.join(save_dir, f'{model_name}.pth'))
            checkpoint_status = ' [Saved Best Model]'
        print(f'Epoch {epoch}: T-Loss: {t_loss:.4f} | V-Loss: {v_loss:.4f} | IoU: {v_iou:.4f} | F1: {v_f1:.4f}{checkpoint_status}')
        early_stopper(v_loss)
        if early_stopper.early_stop:
            print(f'\nEarly stopping triggered at epoch {epoch}. Stopping training.')
            break
    return history


In [ ]:
def extract_one_batch_features(model, dataloader, target_layer, device):
    activations = []

    def hook_fn(module, inputs, output):
        activations.append(output.detach().cpu())
    hook = target_layer.register_forward_hook(hook_fn)
    model.to(device)
    model.eval()
    images, masks = next(iter(dataloader))
    with torch.no_grad():
        _ = model(images.to(device))
    hook.remove()
    if not activations:
        raise RuntimeError('The feature hook did not capture any activations.')
    features = activations[0]
    print('images:', tuple(images.shape))
    print('masks:', tuple(masks.shape))
    print('frozen encoder features:', tuple(features.shape))
    return (images, masks, features)


In [ ]:
NUM_EPOCHS = 100
BATCH_SIZE = 16
IMG_SIZE = 512
USE_CUSTOM_STATS = False
RUN_FEATURE_MAP_SANITY_CHECK = True
all_results = []
last_run = {}
for config, name in tqdm(experiments, desc='Running TTD frozen-encoder feature-extraction experiments'):
    print('\n' + '=' * 90)
    print(f'Experiment: {name}')
    train_dl, val_dl, test_dl, custom_stats = baseline_model_pipeline(base_dir=base_dir, dict_files=config, bs=BATCH_SIZE, img_size=IMG_SIZE, use_custom_stats=USE_CUSTOM_STATS)
    model = build_feature_extraction_unet('resnet34', encoder_weights='imagenet', classes=2)
    model_name = f'Unet-resnet34-imagenet-FE_{name}'
    print('\nModel:', model_name)
    parameter_report(model)
    assert_encoder_is_frozen(model)
    if RUN_FEATURE_MAP_SANITY_CHECK:
        target_layer = model.encoder.layer4[-1]
        _images, _masks, _features = extract_one_batch_features(model, train_dl, target_layer, device)
    optimizer = optim.AdamW(trainable_parameters(model), lr=0.0001)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-07)
    weights = torch.tensor([1.0, 20.0], dtype=torch.float32, device=device)
    loss_fn = TTDCombinedLoss(ce_weight_tensor=weights, w_ce=0.5, w_dice=0.5)
    history = epochs_feature_extractor(model=model, model_name=model_name, device=device, train_dl=train_dl, val_dl=val_dl, loss_fn=loss_fn, optimizer=optimizer, num_epoch=NUM_EPOCHS, scheduler=scheduler, patience=10, save_dir='models')
    save_training_history(history, model_name, save_dir='figures')
    checkpoint_path = os.path.join('models', f'{model_name}.pth')
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()
    v_loss, v_iou, v_f1, v_recall, v_prec = val_loop(model, device, val_dl, loss_fn, is_test=True)
    t_loss, t_iou, t_f1, t_recall, t_prec = val_loop(model, device, test_dl, loss_fn, is_test=True)
    save_prediction_overlap(model, model_name, test_dl, device, custom_stats=custom_stats, save_dir='figures')
    all_results.append({'Experiment': model_name, 'Frozen_Encoder': True, 'Val_Loss': v_loss, 'Val_IoU': v_iou, 'Val_F1': v_f1, 'Val_Recall': v_recall, 'Val_Prec': v_prec, 'Test_Loss': t_loss, 'Test_IoU': t_iou, 'Test_F1': t_f1, 'Test_Recall': t_recall, 'Test_Prec': t_prec})
    results_df = pd.DataFrame(all_results)
    results_df.to_csv('TTD_baseline_fe_results.csv', index=False)
    last_run = {'model': model, 'model_name': model_name, 'train_dl': train_dl, 'val_dl': val_dl, 'test_dl': test_dl, 'custom_stats': custom_stats, 'loss_fn': loss_fn, 'history': history}
results_df = pd.DataFrame(all_results)
display(results_df)
results_df.to_csv('TTD_baseline_fe_results.csv', index=False)
